# Real-Time Building Energy Prediction with Spark Structured Streaming


## Overview

This notebook implements the **real-time streaming inference pipeline**. It subscribes to a Kafka topic populated by the weather data producer (notebook 02), enriches each record with building metadata, applies the pre-trained GBT pipeline from notebook 01, and persists predictions in three aggregation formats.

### Pipeline Position

```
Kafka Topic: weather-stream --> [03 Spark Streaming] --> output/predictions
                                         ^               output/hourly_aggregations
                             models/energy_pipeline_model  output/daily_aggregations
                             (trained in Notebook 01)
```

### Dependencies

| Dependency | Required state |
|------------|----------------|
| **Notebook 01** | `models/energy_pipeline_model` must exist |
| **Notebook 02** | Must be actively publishing to `weather-stream` |
| **Docker** | Kafka broker running at `kafka:9092` |

### Outputs

| Path | Description |
|------|-------------|
| `output/predictions` | Raw per-record predictions (Parquet, append) |
| `output/hourly_aggregations` | Total energy per building per 6-hour window |
| `output/daily_aggregations` | Total energy per site per day |


In [1]:
import os
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark import SparkContext
from pyspark import SparkConf
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.ml.feature import Imputer
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pyspark.ml.feature import OneHotEncoder, StringIndexer
import math
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import PipelineModel
from pyspark.sql.functions import to_json, struct

## 1. Spark Session Setup

Initialise a local SparkSession with 4 cores and the Kafka integration packages loaded via `PYSPARK_SUBMIT_ARGS`. The session timezone is set to Melbourne to align event-time windows with the source data, and a checkpoint base path is defined for all streaming queries.


In [2]:
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.4.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0 pyspark-shell'

# Create Spark session
app_name = "building-energy-streaming"
checkpoint = "checkpoint/streaming"
spark = (
    SparkSession.builder.appName(app_name)
    .master("local[4]") # 4 cores
    .config("spark.sql.session.timeZone", "Australia/Melbourne") # Melbourne time zone
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

#get sparkcontext from the sparksession
sc = spark.sparkContext

26/06/24 14:58:00 WARN Utils: Your hostname, yundeMacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.213.55 instead (on interface en0)
26/06/24 14:58:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/yun/.ivy2/cache
The jars for the packages stored in: /Users/yun/.ivy2/jars
org.apache.spark#spark-streaming-kafka-0-10_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f8ede807-620a-46e9-8b87-7b1da5344bd1;1.0
	confs: [default]
	found org.apache.spark#spark-streaming-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-clien

:: loading settings :: url = jar:file:/Users/yun/projects/building-energy-spark-pipeline/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 146ms :: artifacts dl 4ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.kafka#kafka-clients;3.3.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 from central in [default]
	org.apache.spark#spark-streaming-kafka-0-10_2.12;3.4.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.slf4j#slf4j-api;2.0.6 from central in [default]
	org.xerial.snappy#snappy-java;1.1.9.1 from central in [default]
	-----------------------------------------------

## 2. Schema Definitions and Static Data

Define typed schemas for:
- **`buildings_schema`** - static building attributes loaded from CSV
- **`weather_row_schema`** - a single weather record as received from the Kafka producer
- **`weather_schema`** - the outer Kafka message envelope (`ts` + array of weather rows)

The building DataFrame is loaded once as a static lookup table and joined into the stream in the next section.


In [3]:
# buildings.csv
buildings_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("building_id", IntegerType(), True),
    StructField("primary_use", StringType(), True),
    StructField("square_feet", IntegerType(), True),
    StructField("floor_count", IntegerType(), True),
    StructField("row_id", IntegerType(), True),
    StructField("year_built", IntegerType(), True),
    StructField("latent_y", DoubleType(), True),
    StructField("latent_s", DoubleType(), True),
    StructField("latent_r", DoubleType(), True)
])

In [4]:
# weather.csv
weather_row_schema = StructType([
    StructField("site_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("air_temperature", StringType(), True),
    StructField("cloud_coverage", StringType(), True),
    StructField("dew_temperature", StringType(), True),
    StructField("sea_level_pressure", StringType(), True),
    StructField("wind_direction", StringType(), True),
    StructField("wind_speed", StringType(), True),
    StructField("weather_ts", StringType(), True)
])

weather_schema = StructType([
    StructField("ts", StringType(), True),
    StructField("rows", ArrayType(weather_row_schema), True)
])

In [5]:
# json file from kafka producer
kafka_schema = StructType([
    StructField("ts", StringType(), True),
    StructField("rows", ArrayType(weather_schema), True)
])

In [6]:
# buildings
buildings_df = spark.read.csv("data/new_building_information.csv",header=True,schema=buildings_schema)
print("Buildings DataFrame Schema:")
buildings_df.printSchema()

Buildings DataFrame Schema:
root
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- floor_count: integer (nullable = true)
 |-- row_id: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_y: double (nullable = true)
 |-- latent_s: double (nullable = true)
 |-- latent_r: double (nullable = true)



## 3. Kafka Stream Ingestion

Subscribe to the `weather-stream` Kafka topic and parse the incoming JSON payload into the defined schema. String fields are cast to their correct numeric types, and the stream is joined with the static building DataFrame on `site_id`.


In [7]:
# configuration
hostip = "localhost"
topic = 'weather-stream'
# load data from Kafka topic
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", f"{hostip}:9092") \
    .option("subscribe", topic) \
    .load()
# converting the key/value to string
df = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
# transformed into the metadata file schema
df = df.select(F.from_json(F.col("value"), weather_schema).alias("data"))
df = df.selectExpr("data.ts as batch_ts", "explode(data.rows) as row")
df = df.select("batch_ts", "row.*")

# transformed into the proper formats
numeric_cols_double = [
    "air_temperature", "dew_temperature",
    "sea_level_pressure", "wind_speed"
]

numeric_cols_int = ["site_id", "cloud_coverage", "wind_direction"]

for col_name in numeric_cols_double:
    df = df.withColumn(
        col_name,
        F.when(F.col(col_name) == "", None)
         .otherwise(F.col(col_name).cast("double"))
    )

for col_name in numeric_cols_int:
    df = df.withColumn(
        col_name,
        F.when(F.col(col_name) == "", None)
         .otherwise(F.col(col_name).cast("int"))
    )

In [8]:
# join building dataframe
feature_df = (df.join(buildings_df, on="site_id", how="left"))

## 4. Event Time and Watermarking

Convert the `weather_ts` Unix timestamp (carried in the Kafka payload) to a Spark `TimestampType` column named `event_time`. A 5-second watermark is applied so that Spark can safely clean up state: any record arriving more than 5 seconds after its event time is discarded.


In [9]:
# create timestamp column from weather_ts
feature_df = feature_df.withColumn("event_time",F.to_timestamp(F.col("weather_ts").cast("double")))

# watermark for 5 sec
feature_df = feature_df.withWatermark("event_time", "5 seconds")

## 5. Feature Engineering

Apply the same transformations used to train the batch ML model in notebook 01, so the streaming DataFrame schema matches the pipeline's expected feature vector:

- **Log transform** `square_feet` -> `log_square_feet`
- **Cyclical encoding** of `time_interval`, `wind_direction`, and `month` (sine + cosine)
- **Decade** derived from `year_built`
- **Zero-fill** any residual nulls introduced by the stream join


In [10]:
# reuse code from notebook 01
feature_df = (feature_df.withColumn("year", F.year("event_time"))
                        .withColumn("month", F.month("event_time"))
                        .withColumn("day", F.dayofmonth("event_time"))
                        .withColumn("time_interval", F.hour("event_time")))

# cyclical encoding
# time_interval
# transfer to numerical 
feature_df = feature_df.withColumn(
    "time_interval_num",
    F.when(F.col("time_interval") == "0:00-5:59", 0)
     .when(F.col("time_interval") == "6:00-11:59", 1)
     .when(F.col("time_interval") == "12:00-17:59", 2)
     .when(F.col("time_interval") == "18:00-23:59", 3)
     .otherwise(None)
)

feature_df = feature_df.withColumn("time_sin",F.sin(2 * math.pi * F.col("time_interval_num") / 4))
feature_df = feature_df.withColumn("time_cos", F.cos(2 * math.pi * F.col("time_interval_num") / 4))

# wind_direction
feature_df = feature_df.withColumn("wind_dir_sin", F.sin(2 * math.pi * F.col("wind_direction")/360))
feature_df = feature_df.withColumn("wind_dir_cos", F.cos(2 * math.pi * F.col("wind_direction")/360))
# date features
# drop year (only 2022)
feature_df = feature_df.drop("year")

# month cyclical encoding
feature_df = feature_df.withColumn("month_sin", F.sin(2 * math.pi * F.col("month")/12))
feature_df = feature_df.withColumn("month_cos", F.cos(2 * math.pi * F.col("month")/12))

# drop day
feature_df = feature_df.drop("day")

# group year_built by decades
feature_df = feature_df.withColumn("decade", (F.col("year_built")/10).cast("int")*10)
feature_df = feature_df.withColumn("log_square_feet", F.log1p(F.col("square_feet")))

cols_to_fill = [
    "building_id","site_id","wind_direction","decade",
    "air_temperature","cloud_coverage","dew_temperature","sea_level_pressure",
    "wind_speed","log_square_feet","floor_count","latent_y","latent_s","latent_r",
    "time_sin","time_cos","wind_dir_sin","wind_dir_cos","month_sin","month_cos"
]

# fill missing value for safety
feature_df = feature_df.fillna(0, subset=cols_to_fill)

## 6. Model Inference and Streaming Queries

Load the saved GBT pipeline and apply it to the live feature stream. Three queries demonstrate different output modes and trigger intervals.


In [11]:
# apply the saved GBT pipeline
pipeline = PipelineModel.load("models/energy_pipeline_model")
pred_stream = pipeline.transform(feature_df)

# --- helpers so this notebook produces real output when run unattended ---
# Streaming queries below are started, then inspected, then stopped. Give each
# query a few micro-batches to populate its sink before we read it, so the
# show()/read() cells contain actual rows rather than an empty result.
import time

def wait_for_table(name, timeout=45, interval=2):
    """Poll a memory-sink table until it has at least one row (or timeout)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if spark.sql(f"SELECT count(*) AS c FROM {name}").first()["c"] > 0:
                return
        except Exception:
            pass
        time.sleep(interval)

def wait_for_path(path, timeout=45, interval=2):
    """Poll a parquet output path until it contains at least one row (or timeout)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if spark.read.parquet(path).limit(1).count() > 0:
                return
        except Exception:
            pass
        time.sleep(interval)

### 6a. Per-Record Predictions (Memory Sink)

Write raw predictions to an in-memory table (`pred_table`) for immediate inspection. Triggered every 5 seconds in append mode.


In [12]:
# create a streaming query to write prediction results into a memory table
query_pred = (pred_stream
         .select("event_time", "site_id", "building_id", "prediction") # select the needed column
         .writeStream
         .outputMode("append")
         .format("memory")
         .queryName("pred_table")
         .trigger(processingTime="5 seconds")
         .option("checkpointLocation", f"{checkpoint}/parquet_pred") 
         .start())

In [13]:
# check the result (wait for a few micro-batches first)
wait_for_table("pred_table")
spark.sql("SELECT * FROM pred_table").show(20, truncate=False)

+-------------------+-------+-----------+------------------+
|event_time         |site_id|building_id|prediction        |
+-------------------+-------+-----------+------------------+
|2026-06-24 16:58:08|5      |708        |1558.2028009581115|
|2026-06-24 16:58:08|5      |705        |259.4514957613663 |
|2026-06-24 16:58:08|5      |704        |259.4514957613663 |
|2026-06-24 16:58:08|5      |702        |259.4514957613663 |
|2026-06-24 16:58:08|5      |701        |1558.2028009581115|
|2026-06-24 16:58:08|5      |699        |1558.2028009581115|
|2026-06-24 16:58:08|5      |741        |259.4514957613663 |
|2026-06-24 16:58:08|5      |739        |1558.2028009581115|
|2026-06-24 16:58:08|5      |736        |1558.2028009581115|
|2026-06-24 16:58:08|5      |734        |259.4514957613663 |
|2026-06-24 16:58:08|5      |723        |1558.2028009581115|
|2026-06-24 16:58:08|5      |719        |259.4514957613663 |
|2026-06-24 16:58:08|5      |711        |259.4514957613663 |
|2026-06-24 16:58:08|5  

In [14]:
# stop the query
query_pred.stop()

### 6b. 6-Hour Window Aggregation per Building

Aggregate total predicted energy consumption over tumbling 6-hour windows, grouped by `building_id`. Uses `complete` output mode and triggers every 7 seconds.


In [15]:
# 6b
# 6 hour window aggregation
energy_window_df = (
    pred_stream
    .groupBy(F.window(F.col("event_time"), "6 hours"), F.col("building_id"))
    .agg(F.sum("prediction").alias("total_energy"))
)

In [16]:
# create the query to see the total energy for each building
query_mem_6h = (
    energy_window_df
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("energy_table")
    .option("checkpointLocation", f"{checkpoint}/mem_6h")
    .trigger(processingTime="7 seconds")
    .start()
)

In [17]:
# check the result (wait for a few micro-batches first)
wait_for_table("energy_table")
spark.sql("SELECT * FROM energy_table").show(20, truncate=False)

+------------------------------------------+-----------+-------------------+
|window                                    |building_id|total_energy       |
+------------------------------------------+-----------+-------------------+
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|68         |8445.74462913001   |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|1080       |120745.12018813108 |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|669        |824.7239844233     |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|1093       |9196.358798793168  |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|508        |18914.54038471261  |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|280        |10466.871653666509 |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|723        |12465.622407664892 |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|382        |7547.870981783636  |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|495        |7021.4210515708555 |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|398        |17937.8814233576   |

In [18]:
# stop the query
query_mem_6h.stop()

### 6c. Daily Aggregation per Site

Aggregate total predicted energy consumption over tumbling 1-day windows, grouped by `site_id`. Uses `complete` output mode and triggers every 14 seconds.


In [19]:
daily_energy_df = (
    pred_stream
    .groupBy( F.window(F.col("event_time"), "1 day"), F.col("site_id"))
    .agg(F.sum("prediction").alias("daily_total_energy"))
)

In [20]:
# create the query to see the daily energy consumption for each site
query_mem_daily = (
    daily_energy_df
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("daily_table")
    .option("checkpointLocation", f"{checkpoint}/mem_daily")
    .trigger(processingTime="14 seconds")
    .start()
)

In [21]:
# check the result (wait for a few micro-batches first)
wait_for_table("daily_table", timeout=60)
spark.sql("SELECT * FROM daily_table").show(truncate=False)

+------------------------------------------+-------+------------------+
|window                                    |site_id|daily_total_energy|
+------------------------------------------+-------+------------------+
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|2      |277401.8483296585 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|9      |1833148.9282041325|
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|0      |496862.1056220831 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|7      |414710.7260120769 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|4      |389776.23966399534|
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|3      |1341243.0030849923|
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|5      |153160.260221196  |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|12     |160160.41075995594|
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|8      |37513.56545568192 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|15     |1579813.2128542976|
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|11     |20702.765141

In [22]:
# stop the query
query_mem_daily.stop()

## 7. Persist Streams to Parquet

Write each of the three result streams to Parquet on disk. Files are updated incrementally with each micro-batch, making them available for downstream batch reads or further streaming queries.


### 7a. Raw Predictions

Append-mode stream writing raw predictions to `output/predictions`. Triggered every 5 seconds.


In [23]:
parquet_query_predictions = (
    pred_stream.select("event_time", "site_id", "building_id", "prediction").writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "output/predictions")
    .option("checkpointLocation", f"{checkpoint}/predictions_stream")
    .trigger(processingTime="5 seconds")
    .start()
)

In [24]:
wait_for_path("output/predictions")
df_predictions = spark.read.parquet("output/predictions")
df_predictions.show(5, truncate=False)

+-------------------+-------+-----------+------------------+
|event_time         |site_id|building_id|prediction        |
+-------------------+-------+-----------+------------------+
|2026-06-24 16:58:33|15     |1395       |2573.3092082890607|
|2026-06-24 16:58:33|15     |1353       |1215.3510911257436|
|2026-06-24 16:58:33|15     |1443       |1149.544849849146 |
|2026-06-24 16:58:33|15     |1441       |1149.544849849146 |
|2026-06-24 16:58:33|15     |1439       |1149.544849849146 |
+-------------------+-------+-----------+------------------+
only showing top 5 rows



In [25]:
# stop the query
parquet_query_predictions.stop()

### 7b. 6-Hour Aggregations

Uses `foreachBatch` to overwrite `output/hourly_aggregations` with the latest complete window state. Triggered every 7 seconds.


In [26]:
def write_to_parquet_hourly(batch_df, batch_id):
    (batch_df
        .write
        .mode("overwrite")
        .parquet(f"output/hourly_aggregations")
    )

parquet_query_hourly = (
    energy_window_df.writeStream
    .foreachBatch(write_to_parquet_hourly)
    .outputMode("complete")
    .trigger(processingTime="7 seconds")
    .option("checkpointLocation", f"{checkpoint}/hourly")
    .start()
)

In [27]:
wait_for_path("output/hourly_aggregations")
df_hourly = spark.read.parquet("output/hourly_aggregations")
df_hourly.show(5, truncate=False)

+------------------------------------------+-----------+------------------+
|window                                    |building_id|total_energy      |
+------------------------------------------+-----------+------------------+
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|973        |44086.01743964542 |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|1157       |9722.808729005948 |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|1282       |9722.808729005948 |
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|327        |15826.854660442525|
|{2026-06-24 16:00:00, 2026-06-24 22:00:00}|914        |44612.46736985819 |
+------------------------------------------+-----------+------------------+
only showing top 5 rows



In [28]:
# stop the query
parquet_query_hourly.stop()

### 7c. Daily Aggregations

Uses `foreachBatch` to overwrite `output/daily_aggregations` with the latest complete window state. Triggered every 14 seconds.


In [29]:
def write_to_parquet_daily(batch_df, batch_id):
    (batch_df
        .write
        .mode("overwrite")
        .parquet(f"output/daily_aggregations")
    )

parquet_query_daily = (
    daily_energy_df.writeStream
    .foreachBatch(write_to_parquet_daily)
    .outputMode("complete")
    .trigger(processingTime="14 seconds")
    .option("checkpointLocation", f"{checkpoint}/daily")
    .start()
)

In [30]:
wait_for_path("output/daily_aggregations", timeout=60)
df_daily= spark.read.parquet("output/daily_aggregations")
df_daily.show(truncate=False)

26/06/24 14:58:56 ERROR Executor: Exception in task 0.0 in stage 120.0 (TID 1505)
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:322)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:396)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:422)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:472)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:464)
	at org.apache.spark.sql.execution.datasources.SchemaMergeUtils$.$anonfun$mergeSchemasInParallel$2(SchemaMergeUtils.scala:79)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2(RDD.scala:853)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2$adapted(RDD.scala:853)
	at org.apache.spark.r

+------------------------------------------+-------+------------------+
|window                                    |site_id|daily_total_energy|
+------------------------------------------+-------+------------------+
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|13     |4556458.988270311 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|10     |633149.7808610402 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|1      |346760.5170976182 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|11     |39586.73174348488 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|15     |1485944.604035632 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|0      |810701.4013140239 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|8      |24525.76436198773 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|9      |3483359.2996967854|
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|6      |720425.5979431411 |
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|3      |2441829.9797899793|
|{2026-06-24 10:00:00, 2026-06-25 10:00:00}|12     |343200.88019

In [31]:
# stop the query
parquet_query_daily.stop()

## 8. Re-publish Results to Kafka

Read each Parquet output directory as a streaming source and forward records to a dedicated Kafka topic as JSON, making results available to any downstream consumer.

| Parquet source | Kafka topic |
|----------------|-------------|
| `output/predictions` | `predictions-stream` |
| `output/hourly_aggregations` | `hourly-stream` |
| `output/daily_aggregations` | `daily-stream` |


### 8a. Raw Predictions -> `predictions-stream`


In [32]:
# Stream 1

#  Read Parquet files as streaming source
parquet_stream_8a = (
    spark.readStream
    .schema(pred_stream.schema)
    .option("maxFilesPerTrigger", 1)
    .parquet("output/predictions")
)

# Convert to JSON for Kafka
kafka_ready_8a = parquet_stream_8a.selectExpr("CAST(null AS STRING) AS key","to_json(struct(*)) AS value")

# Send to Kafka
kafka_query_8a = (
    kafka_ready_8a.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", "predictions-stream")
    .option("checkpointLocation", f"{checkpoint}/kafka_8a")
    .outputMode("append")
    .start()
)

# let at least one micro-batch publish before we stop it
time.sleep(8)
print("predictions-stream query active:", kafka_query_8a.isActive,
      "| last progress rows:", (kafka_query_8a.lastProgress or {}).get("numInputRows"))

predictions-stream query active: True | last progress rows: 3422


In [33]:
# stop the query
kafka_query_8a.stop()

### 8b. 6-Hour Aggregations -> `hourly-stream`


In [34]:
# Stream 2

#  Read Parquet files as streaming source
parquet_stream_8b = (
    spark.readStream
    .schema(pred_stream.schema)
    .option("maxFilesPerTrigger", 1)
    .parquet("output/hourly_aggregations")
)

# Convert to JSON for Kafka
kafka_ready_8b = parquet_stream_8b.selectExpr(
    "CAST(null AS STRING) AS key",
    "to_json(struct(*)) AS value"
)

# Send to Kafka
kafka_query_8b = (
    kafka_ready_8b.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", "hourly-stream")
    .option("checkpointLocation", f"{checkpoint}/kafka_8b")
    .outputMode("append")
    .start()
)

# let at least one micro-batch publish before we stop it
time.sleep(8)
print("hourly-stream query active:", kafka_query_8b.isActive,
      "| last progress rows:", (kafka_query_8b.lastProgress or {}).get("numInputRows"))

hourly-stream query active: True | last progress rows: 2


In [35]:
# stop the query
kafka_query_8b.stop()

26/06/24 14:59:16 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 90, writer: org.apache.spark.sql.kafka010.KafkaStreamingWrite@32ae742b] is aborting.
26/06/24 14:59:16 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 90, writer: org.apache.spark.sql.kafka010.KafkaStreamingWrite@32ae742b] aborted.


### 8c. Daily Aggregations -> `daily-stream`


In [36]:
# Stream 3

#  Read Parquet files as streaming source
parquet_stream_8c = (
    spark.readStream
    .schema(pred_stream.schema)
    .option("maxFilesPerTrigger", 1)
    .parquet("output/daily_aggregations")
)

# Convert to JSON for Kafka
kafka_ready_8c = parquet_stream_8c.selectExpr(
    "CAST(null AS STRING) AS key",
    "to_json(struct(*)) AS value"
)

# Send to Kafka
kafka_query_8c = (
    kafka_ready_8c.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", "daily-stream")
    .option("checkpointLocation", f"{checkpoint}/kafka_8c")
    .outputMode("append")
    .start()
)

# let at least one micro-batch publish before we stop it
time.sleep(8)
print("daily-stream query active:", kafka_query_8c.isActive,
      "| last progress rows:", (kafka_query_8c.lastProgress or {}).get("numInputRows"))

daily-stream query active: True | last progress rows: 1


In [37]:
# stop the query
kafka_query_8c.stop()